# 6.2 - Predicting LLM Disagreement From Text Alone

This notebook is the simplified, stable version of the disagreement-prediction analysis.

It compares just two text representations:
- a **Gorodnichenko-style monetary-policy dictionary** from `literature/Gorodnichenko.md`
- **TF-IDF**

The goal is to predict LLM disagreement on a given turn using meeting-grouped cross-validation only.

## Targets

We keep two disagreement targets:

1. `split` — **headline target**. A turn is a split if at least one LLM is neutral and at least one other LLM is directional.
2. `high_std3` — **secondary target**. A binary indicator for whether `score_std_3way` falls in the high-instability tail of the distribution.

This target choice matches the main substantive pattern in the project: disagreement is mostly about the **neutral vs. stanced boundary**, while `high_std3` captures unusually large instability beyond the binary split. Both targets are now evaluated with ROC-AUC.


In [ ]:
from __future__ import annotations

import math
import re
from itertools import combinations
from pathlib import Path

import matplotlib
from IPython import get_ipython

ip = get_ipython()
if ip is None:
    matplotlib.use("Agg")
elif "IPKernelApp" in getattr(ip, "config", {}):
    ip.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm, spearmanr
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, Ridge, RidgeCV
from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
plt.style.use("ggplot")

IN_NOTEBOOK = ip is not None and "IPKernelApp" in getattr(ip, "config", {})

def finalize_plot(close: bool = True) -> None:
    if IN_NOTEBOOK:
        plt.show()
    if close:
        plt.close()


In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "output").exists() and (ROOT.parent / "output").exists():
    ROOT = ROOT.parent
elif not (ROOT / "output").exists() and (Path("..").resolve() / "output").exists():
    ROOT = Path("..").resolve()

OUTPUT_DIR = ROOT / "output" / "stance"
FIGURE_DIR = ROOT / "output" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    "deepseekv3": OUTPUT_DIR / "turn_predictions_deepseekv3.csv",
    "gemini25flash": OUTPUT_DIR / "turn_predictions_gemini25flash.csv",
    "gpt-4o": OUTPUT_DIR / "turn_predictions_gpt-4o.csv",
    "llama33": OUTPUT_DIR / "turn_predictions_llama33.csv",
    "mistrallarge_or": OUTPUT_DIR / "turn_predictions_mistrallarge_or.csv",
    "qwen25_72b": OUTPUT_DIR / "turn_predictions_qwen25_72b.csv",
}
MERGED_TARGET_PATH = OUTPUT_DIR / "turn_disagreement_targets.csv"
FOLD_PATH = OUTPUT_DIR / "disagreement_simple_cv_folds.csv"
SUMMARY_PATH = OUTPUT_DIR / "disagreement_simple_summary.csv"
PRAUC_TABLE_PATH = OUTPUT_DIR / "disagreement_simple_pr_auc_summary.csv"
FOLD_METRICS_PATH = OUTPUT_DIR / "disagreement_simple_fold_metrics.csv"
POOLED_PRED_PATH = OUTPUT_DIR / "disagreement_simple_pooled_predictions.csv"
REG_SUMMARY_PATH = OUTPUT_DIR / "disagreement_score_std3_regression_summary.csv"
REG_PRED_PATH = OUTPUT_DIR / "disagreement_score_std3_regression_predictions.csv"
TFIDF_FEATURES_PATH = OUTPUT_DIR / "disagreement_simple_tfidf_features.csv"
TFIDF_SHAP_STYLE_PATH = OUTPUT_DIR / "disagreement_simple_tfidf_shap_style.csv"
MECH_TABLE_PATH = OUTPUT_DIR / "disagreement_simple_mechanism_logit.csv"
MECH_EXPOSURE_PATH = OUTPUT_DIR / "disagreement_simple_mechanism_exposures.csv"
MECH_EXPOSURE_TEST_PATH = OUTPUT_DIR / "disagreement_simple_mechanism_tests.csv"
INFLATION_FRAME_PATH = OUTPUT_DIR / "disagreement_inflation_frames.csv"
INFLATION_MODEL_PATH = OUTPUT_DIR / "disagreement_inflation_by_model.csv"
INFLATION_DOSERESP_PATH = OUTPUT_DIR / "disagreement_inflation_dose_response.csv"
INFLATION_MODEL_DOSERESP_PATH = OUTPUT_DIR / "disagreement_inflation_model_dose_response.csv"
INFLATION_MODEL_CENTERED_PATH = OUTPUT_DIR / "disagreement_inflation_model_centered.csv"
HIGH_STD3_QUANTILE = 0.75

LLMS = list(INPUT_FILES.keys())
N_SPLITS_CV = 5
RANDOM_STATE = 42

LABEL_SCORE_3WAY = {
    "dovish": -1,
    "mostly dovish": -1,
    "neutral": 0,
    "mostly hawkish": 1,
    "hawkish": 1,
}
STANCED = {"dovish", "mostly dovish", "mostly hawkish", "hawkish"}

meta_cols = [
    "turn_uid",
    "bank",
    "doc_id",
    "date",
    "doc_type",
    "speaker",
    "speaker_role",
    "turn_idx",
    "text",
]

print(f"Root: {ROOT}")
print(f"Merged target path: {MERGED_TARGET_PATH}")
print(f"Fold path: {FOLD_PATH}")
print(f"High-std3 quantile: {HIGH_STD3_QUANTILE:.2f}")


## 1. Load or Build the Disagreement Dataset

The raw LLM labels are collapsed to a 3-way ordinal scale:
- dovish / mostly dovish -> `-1`
- neutral -> `0`
- mostly hawkish / hawkish -> `+1`

This lets us define both the binary boundary-disagreement target and the high-instability target on the same scale.


In [ ]:
def build_disagreement_from_raw() -> pd.DataFrame:
    frames = []
    for model_key, path in INPUT_FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing raw input file: {path}")
        df = pd.read_csv(path)
        df["label"] = df["label"].astype(str).str.strip().str.lower()
        keep = df[meta_cols + ["label"]].copy().rename(columns={"label": f"label_{model_key}"})
        frames.append(keep)

    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(frame, on=meta_cols, how="inner")

    label_cols = [c for c in merged.columns if c.startswith("label_")]
    score3_cols = []
    stanced_cols = []
    for model_key in LLMS:
        score_col = f"score3_{model_key}"
        stanced_col = f"stanced_{model_key}"
        merged[score_col] = merged[f"label_{model_key}"].map(LABEL_SCORE_3WAY)
        merged[stanced_col] = merged[f"label_{model_key}"].isin(STANCED)
        score3_cols.append(score_col)
        stanced_cols.append(stanced_col)

    merged["n_models"] = len(LLMS)
    merged["n_stanced"] = merged[stanced_cols].sum(axis=1)
    merged["split"] = ((merged["n_stanced"] > 0) & (merged["n_stanced"] < len(LLMS))).astype(int)
    merged["p_directional"] = merged["n_stanced"] / len(LLMS)
    merged["score_std_3way"] = merged[score3_cols].std(axis=1, ddof=0)
    merged["mean_score_3way"] = merged[score3_cols].mean(axis=1)
    merged["sign_conflict"] = (
        merged[score3_cols].eq(-1).any(axis=1) & merged[score3_cols].eq(1).any(axis=1)
    ).astype(int)
    merged["text"] = merged["text"].astype(str)
    return merged.sort_values(["doc_id", "turn_idx", "turn_uid"]).reset_index(drop=True)


needed = {"split", "score_std_3way", "doc_id", "text", "sign_conflict", "n_stanced"}
if MERGED_TARGET_PATH.exists():
    disagreement = pd.read_csv(MERGED_TARGET_PATH)
    if not needed.issubset(disagreement.columns):
        disagreement = build_disagreement_from_raw()
        disagreement.to_csv(MERGED_TARGET_PATH, index=False)
        source_used = f"rebuilt merged file -> {MERGED_TARGET_PATH}"
    else:
        source_used = f"loaded merged file -> {MERGED_TARGET_PATH}"
else:
    disagreement = build_disagreement_from_raw()
    disagreement.to_csv(MERGED_TARGET_PATH, index=False)
    source_used = f"built merged file -> {MERGED_TARGET_PATH}"

disagreement["text"] = disagreement["text"].astype(str)
disagreement["date"] = disagreement["date"].astype(str)
disagreement["date_ym"] = disagreement["date"].str.slice(0, 6).astype(int)
high_std3_threshold = float(disagreement["score_std_3way"].quantile(HIGH_STD3_QUANTILE))
disagreement["high_std3"] = (disagreement["score_std_3way"] >= high_std3_threshold).astype(int)

print(source_used)
print(f"Turns: {len(disagreement):,}")
print(f"Meetings: {disagreement['doc_id'].nunique()}")
print(f"high_std3 threshold (q={HIGH_STD3_QUANTILE:.2f}): {high_std3_threshold:.6f}")
print(f"high_std3 positive share: {disagreement['high_std3'].mean():.1%}")
display(disagreement[["turn_uid", "bank", "doc_id", "split", "score_std_3way", "high_std3", "n_stanced", "sign_conflict"]].head())


## 2. Target Diagnostics

The headline target is `split`, because it directly captures the neutral-vs-stanced boundary where the project's disagreement mostly lives. `high_std3` is the secondary target because it marks turns in the high-instability tail of the same `{-1, 0, +1}` dispersion measure.


In [ ]:
target_summary = pd.DataFrame([
    {
        "target": "split",
        "type": "classification",
        "mean": disagreement["split"].mean(),
        "std": disagreement["split"].std(ddof=0),
        "min": disagreement["split"].min(),
        "max": disagreement["split"].max(),
    },
    {
        "target": "high_std3",
        "type": "classification",
        "mean": disagreement["high_std3"].mean(),
        "std": disagreement["high_std3"].std(ddof=0),
        "min": disagreement["high_std3"].min(),
        "max": disagreement["high_std3"].max(),
    },
])

meeting_df = (
    disagreement.groupby("doc_id", as_index=False)
    .agg(
        bank=("bank", "first"),
        date_ym=("date_ym", "first"),
        n_turns=("turn_uid", "count"),
        mean_split=("split", "mean"),
        mean_std3=("score_std_3way", "mean"),
        share_high_std3=("high_std3", "mean"),
    )
)

print("Target summary")
display(target_summary.style.format({"mean": "{:.3f}", "std": "{:.3f}", "min": "{:.3f}", "max": "{:.3f}"}))

print("Meeting-level summary")
display(
    meeting_df[["bank", "n_turns", "mean_split", "mean_std3", "share_high_std3"]]
    .describe()
    .T
    .style.format("{:.3f}")
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
split_counts = disagreement["split"].map({0: "No split", 1: "Split"}).value_counts()
axes[0].bar(split_counts.index, split_counts.values, color=["#4daf4a", "#e41a1c"])
axes[0].set_title("Headline target: split")
axes[0].set_ylabel("Turns")

high_counts = disagreement["high_std3"].map({0: "Not high-std3", 1: "High-std3"}).value_counts()
axes[1].bar(high_counts.index, high_counts.values, color=["#80b1d3", "#fb8072"])
axes[1].set_title("Secondary target: high_std3")
axes[1].set_ylabel("Turns")

plt.tight_layout()
finalize_plot()

print(f"Split rate: {disagreement['split'].mean():.1%}")
print(f"Mean score_std_3way: {disagreement['score_std_3way'].mean():.3f}")
print(f"High-std3 rate: {disagreement['high_std3'].mean():.1%}")
print(f"Sign-conflict rate (at least one -1 and one +1): {disagreement['sign_conflict'].mean():.1%}")


## 3. Locked Meeting-Level Cross-Validation

All evaluation is at the meeting level. The fold assignment is created once, saved, and reloaded on reruns.


In [ ]:
meeting_df["split_bin"] = pd.qcut(meeting_df["mean_split"], q=3, labels=["low", "med", "high"], duplicates="drop")
meeting_df["strata"] = meeting_df["bank"] + "_" + meeting_df["split_bin"].astype(str)
min_stratum = meeting_df["strata"].value_counts().min()
print("Stratum counts (bank x split bin):")
print(meeting_df.groupby(["bank", "split_bin"]).size().unstack(fill_value=0))
print(f"\nSmallest stratum: {min_stratum}")
assert min_stratum >= N_SPLITS_CV, f"Not enough meetings in smallest stratum for {N_SPLITS_CV}-fold CV"

if FOLD_PATH.exists():
    fold_map = pd.read_csv(FOLD_PATH)
    print(f"Loaded folds from {FOLD_PATH}")
else:
    skf = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
    meeting_df["cv_fold"] = -1
    for fold, (_, test_idx) in enumerate(skf.split(meeting_df, meeting_df["strata"])):
        meeting_df.iloc[test_idx, meeting_df.columns.get_loc("cv_fold")] = fold
    fold_map = meeting_df[["doc_id", "cv_fold"]].copy()
    fold_map.to_csv(FOLD_PATH, index=False)
    print(f"Created folds -> {FOLD_PATH}")

disagreement = disagreement.merge(fold_map, on="doc_id", how="left")
assert disagreement["cv_fold"].notna().all(), "Some turns did not get a fold assignment"
assert disagreement.groupby("doc_id")["cv_fold"].nunique().eq(1).all(), "Meeting leakage across folds"

print("\nMeetings per fold:")
print(fold_map["cv_fold"].value_counts().sort_index().rename("n_meetings"))
print("\nBank x fold distribution:")
display(disagreement[["doc_id", "bank", "cv_fold"]].drop_duplicates().groupby(["cv_fold", "bank"]).size().unstack(fill_value=0))


## 4. Gorodnichenko Dictionary Representation

This implements a simplified version of the phrase logic described in `literature/Gorodnichenko.md`:

- A1 + A2 or B1 + B2 -> dovish phrase
- A1 + B2 or B1 + A2 -> hawkish phrase
- negation flips the sentence classification

The output is not a replacement labeler; it is a fixed, transparent text representation used to predict disagreement. This simplified notebook still does not explicitly isolate mixed hawk/dove cue competition inside the same turn; it only uses the aggregated Gorodnichenko-style features as predictors.


In [ ]:
token_pattern = re.compile(r"\b[a-zA-Z][a-zA-Z\-']+\b")
sentence_splitter = re.compile(r"(?<=[\.!?])\s+")

def tokenize(text: str) -> list[str]:
    return token_pattern.findall(str(text).lower())

A1 = ["inflation expectation", "interest rate", "bank rate", "fund rate", "price", "economic activity", "inflation", "employment"]
A2 = ["anchor", "cut", "subdue", "declin", "decrease", "reduc", "low", "drop", "fall", "fell", "decelerat", "slow", "pause", "pausing", "stable", "non-accelerating", "downward", "tighten"]
B1 = ["unemployment", "growth", "exchange rate", "productivity", "deficit", "demand", "job market", "monetary policy"]
B2 = ["ease", "easing", "rise", "rising", "increase", "expand", "improv", "strong", "upward", "raise", "high", "rapid"]
NEGATIONS = [
    "weren't", "were not", "wasn't", "was not", "did not", "didn't", "do not", "don't", "will not", "won't",
    "cannot", "can't", "no longer", "not", "never"
]

def contains_pattern(sentence: str, patterns: list[str]) -> bool:
    sent = sentence.lower()
    toks = tokenize(sent)
    for pat in patterns:
        if " " in pat:
            if pat in sent:
                return True
        else:
            if any(tok.startswith(pat) for tok in toks):
                return True
    return False

def gorod_sentence_label(sentence: str) -> int:
    has_a1 = contains_pattern(sentence, A1)
    has_a2 = contains_pattern(sentence, A2)
    has_b1 = contains_pattern(sentence, B1)
    has_b2 = contains_pattern(sentence, B2)
    has_neg = contains_pattern(sentence, NEGATIONS)

    dovish = (has_a1 and has_a2) or (has_b1 and has_b2)
    hawkish = (has_a1 and has_b2) or (has_b1 and has_a2)

    if dovish and not hawkish:
        return 1 if has_neg else -1
    if hawkish and not dovish:
        return -1 if has_neg else 1
    return 0

def gorod_features(text: str) -> dict[str, float]:
    text = str(text)
    sentences = [s.strip() for s in sentence_splitter.split(text) if s.strip()]
    if not sentences:
        sentences = [text]

    labels = [gorod_sentence_label(sentence) for sentence in sentences]
    dovish = sum(label == -1 for label in labels)
    hawkish = sum(label == 1 for label in labels)
    neutral = sum(label == 0 for label in labels)
    total = max(len(labels), 1)
    toks = tokenize(text)
    n_tokens = max(len(toks), 1)

    return {
        "goro_hawk_sent": hawkish,
        "goro_dove_sent": dovish,
        "goro_neutral_sent": neutral,
        "goro_net_sent": hawkish - dovish,
        "goro_abs_sent": hawkish + dovish,
        "goro_hawk_share": hawkish / total,
        "goro_dove_share": dovish / total,
        "goro_neutral_share": neutral / total,
        "goro_net_share": (hawkish - dovish) / total,
        "goro_sentence_count": total,
        "goro_token_count": n_tokens,
    }

def build_dict_frame(text_series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame([gorod_features(text) for text in text_series], index=text_series.index)

goro_preview = build_dict_frame(disagreement["text"].head(5))
display(goro_preview)


## 5. Evaluation Setup

We compare:
- **Gorodnichenko dictionary + simple downstream model**
- **TF-IDF + simple downstream model**

Both targets are binary, so both are evaluated with logistic regression and summarized with ROC-AUC / PR-AUC.


In [ ]:
def classification_metrics(y_true, y_pred_proba) -> dict[str, float]:
    return {
        "roc_auc": roc_auc_score(y_true, y_pred_proba),
        "pr_auc": average_precision_score(y_true, y_pred_proba),
    }

def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "spearman": spearmanr(y_true, y_pred).statistic,
    }

# Shared hyperparameter grids — same candidates for both models so regularization is tuned on equal footing.
C_GRID = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
ALPHA_GRID = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]

dict_clf = Pipeline([
    ("scale", StandardScaler()),
    ("logit", LogisticRegressionCV(Cs=C_GRID, cv=3, max_iter=5000, class_weight="balanced", scoring="roc_auc", refit=True)),
])
tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5, stop_words="english", sublinear_tf=True)),
    ("logit", LogisticRegressionCV(Cs=C_GRID, cv=3, max_iter=5000, class_weight="balanced", scoring="roc_auc", refit=True)),
])
dict_reg = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", RidgeCV(alphas=ALPHA_GRID)),
])
tfidf_reg = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5, stop_words="english", sublinear_tf=True)),
    ("ridge", RidgeCV(alphas=ALPHA_GRID)),
])

target_specs = {
    "split": {"type": "classification", "headline": "roc_auc", "secondary": "pr_auc"},
    "high_std3": {"type": "classification", "headline": "roc_auc", "secondary": "pr_auc"},
}


## 6. 5-Fold Meeting-Grouped CV: Gorodnichenko vs TF-IDF


In [ ]:
fold_metrics = []
pooled_predictions = []

for fold in range(N_SPLITS_CV):
    train_df = disagreement[disagreement["cv_fold"] != fold].copy()
    test_df = disagreement[disagreement["cv_fold"] == fold].copy()
    assert set(train_df["doc_id"]).isdisjoint(set(test_df["doc_id"])), f"Meeting leakage in fold {fold}"

    X_dict_train = build_dict_frame(train_df["text"])
    X_dict_test = build_dict_frame(test_df["text"])

    for target, spec in target_specs.items():
        y_train = train_df[target].values
        y_test = test_df[target].values

        dict_model = clone(dict_clf).fit(X_dict_train, y_train)
        dict_pred = dict_model.predict_proba(X_dict_test)[:, 1]
        dict_metrics = classification_metrics(y_test, dict_pred)

        tfidf_model = clone(tfidf_clf).fit(train_df["text"], y_train)
        tfidf_pred = tfidf_model.predict_proba(test_df["text"])[:, 1]
        tfidf_metrics = classification_metrics(y_test, tfidf_pred)

        fold_metrics.append({"fold": fold, "target": target, "model": "Gorodnichenko", **dict_metrics})
        fold_metrics.append({"fold": fold, "target": target, "model": "TF-IDF", **tfidf_metrics})

        pooled_predictions.append(pd.DataFrame({
            "turn_uid": test_df["turn_uid"].values,
            "doc_id": test_df["doc_id"].values,
            "bank": test_df["bank"].values,
            "cv_fold": fold,
            "target": target,
            "y_true": y_test,
            "pred_gorodnichenko": dict_pred,
            "pred_tfidf": tfidf_pred,
        }))

fold_metrics_df = pd.DataFrame(fold_metrics)
pooled_pred_df = pd.concat(pooled_predictions, ignore_index=True)
fold_metrics_df.to_csv(FOLD_METRICS_PATH, index=False)
pooled_pred_df.to_csv(POOLED_PRED_PATH, index=False)

summary_rows = []
for target, spec in target_specs.items():
    for model_name in ["Gorodnichenko", "TF-IDF"]:
        grp = fold_metrics_df[(fold_metrics_df["target"] == target) & (fold_metrics_df["model"] == model_name)]
        preds = pooled_pred_df[pooled_pred_df["target"] == target]
        if model_name == "Gorodnichenko":
            pred_col = "pred_gorodnichenko"
        else:
            pred_col = "pred_tfidf"
        pooled_headline = roc_auc_score(preds["y_true"], preds[pred_col])
        pooled_secondary = average_precision_score(preds["y_true"], preds[pred_col])
        summary_rows.append({
            "target": target,
            "model": model_name,
            "cv_mean_headline": grp[spec["headline"]].mean(),
            "cv_sd_headline": grp[spec["headline"]].std(ddof=1),
            "pooled_headline": pooled_headline,
            "cv_mean_secondary": grp[spec["secondary"]].mean(),
            "pooled_secondary": pooled_secondary,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)

for target, spec in target_specs.items():
    print(f"\n=== {target} ===")
    display(
        summary_df[summary_df["target"] == target]
        .style.format({
            "cv_mean_headline": "{:.3f}",
            "cv_sd_headline": "{:.3f}",
            "pooled_headline": "{:.3f}",
            "cv_mean_secondary": "{:.3f}",
            "pooled_secondary": "{:.3f}",
        })
    )

print(f"Saved fold metrics -> {FOLD_METRICS_PATH}")
print(f"Saved pooled predictions -> {POOLED_PRED_PATH}")
print(f"Saved summary -> {SUMMARY_PATH}")


## 7. Main Comparison Figure


In [ ]:
headline_labels = {"split": "ROC-AUC", "high_std3": "ROC-AUC"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for ax, target in zip(axes, ["split", "high_std3"]):
    sub = summary_df[summary_df["target"] == target].copy()
    colors = ["#8c510a", "#01665e"]
    x = np.arange(len(sub))
    ax.bar(x, sub["pooled_headline"], color=colors, edgecolor="white", linewidth=0.8)
    ax.errorbar(x, sub["cv_mean_headline"], yerr=sub["cv_sd_headline"], fmt="o", color="black", capsize=4, label="CV mean ± SD")
    for i, row in sub.reset_index(drop=True).iterrows():
        ax.text(i, row["pooled_headline"] + 0.015, f"{row['pooled_headline']:.3f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["model"])
    ax.set_title(f"{target} ({headline_labels[target]})", fontsize=12, fontweight="bold")
    ax.set_ylabel(headline_labels[target])
    ax.grid(axis="y", alpha=0.25)
    ax.set_ylim(0.45, 1.0)
    ax.legend(loc="lower right", fontsize=8, frameon=True)

plt.suptitle("Predicting LLM disagreement: Gorodnichenko dictionary vs TF-IDF", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_gorodnichenko_vs_tfidf.png", dpi=160, bbox_inches="tight")
finalize_plot()


## 7b. Prediction Tracking: PR-AUC, Calibration, and Meeting-Level Time Series

ROC-AUC is the headline metric, but for disagreement work it also helps to show how the models perform on the positive class directly and whether text-predicted disagreement tracks realized disagreement in an interpretable way.


In [ ]:
pr_auc_table = (
    summary_df[["target", "model", "cv_mean_secondary", "pooled_secondary"]]
    .rename(columns={"cv_mean_secondary": "cv_mean_pr_auc", "pooled_secondary": "pooled_pr_auc"})
    .copy()
)
pr_auc_table.to_csv(PRAUC_TABLE_PATH, index=False)
print("PR-AUC summary table")
display(pr_auc_table.style.format({"cv_mean_pr_auc": "{:.3f}", "pooled_pr_auc": "{:.3f}"}))


## 7c. Continuous Disagreement Intensity: `score_std_3way` Regression

The `split` target is the best interpretive measure for the neutral-versus-stanced boundary, but it is a weak meeting-level time-series target because one diverging model can flip the indicator. This block therefore predicts the continuous disagreement intensity measure `score_std_3way` directly.


In [ ]:
import time

reg_fold_metrics = []
reg_pooled_predictions = []

for fold in range(N_SPLITS_CV):
    t0 = time.time()
    print(f"Fold {fold+1}/{N_SPLITS_CV} — starting...", flush=True)

    train_df = disagreement[disagreement["cv_fold"] != fold].copy()
    test_df = disagreement[disagreement["cv_fold"] == fold].copy()

    X_dict_train = build_dict_frame(train_df["text"])
    X_dict_test = build_dict_frame(test_df["text"])
    y_train = train_df["score_std_3way"].values
    y_test = test_df["score_std_3way"].values

    dict_model = clone(dict_reg).fit(X_dict_train, y_train)
    dict_pred = np.clip(dict_model.predict(X_dict_test), 0, None)
    dict_metrics = regression_metrics(y_test, dict_pred)
    print(f"  Gorodnichenko done ({time.time()-t0:.1f}s)", flush=True)

    tfidf_model = clone(tfidf_reg).fit(train_df["text"], y_train)
    tfidf_pred = np.clip(tfidf_model.predict(test_df["text"]), 0, None)
    tfidf_metrics = regression_metrics(y_test, tfidf_pred)
    print(f"  TF-IDF done ({time.time()-t0:.1f}s)", flush=True)

    reg_fold_metrics.append({"fold": fold, "model": "Gorodnichenko", **dict_metrics})
    reg_fold_metrics.append({"fold": fold, "model": "TF-IDF", **tfidf_metrics})

    reg_pooled_predictions.append(pd.DataFrame({
        "turn_uid": test_df["turn_uid"].values,
        "doc_id": test_df["doc_id"].values,
        "bank": test_df["bank"].values,
        "date": test_df["date"].values,
        "cv_fold": fold,
        "y_true_std3": y_test,
        "pred_std3_gorodnichenko": dict_pred,
        "pred_std3_tfidf": tfidf_pred,
    }))
    print(f"  Fold {fold+1} complete ({time.time()-t0:.1f}s)", flush=True)

reg_fold_metrics_df = pd.DataFrame(reg_fold_metrics)
reg_pred_df = pd.concat(reg_pooled_predictions, ignore_index=True)
reg_pred_df.to_csv(REG_PRED_PATH, index=False)

reg_summary_rows = []
for model_name, pred_col in [("Gorodnichenko", "pred_std3_gorodnichenko"), ("TF-IDF", "pred_std3_tfidf")]:
    grp = reg_fold_metrics_df[reg_fold_metrics_df["model"] == model_name]
    pooled_metrics = regression_metrics(reg_pred_df["y_true_std3"], reg_pred_df[pred_col])
    reg_summary_rows.append({
        "model": model_name,
        "cv_mean_r2": grp["r2"].mean(),
        "cv_sd_r2": grp["r2"].std(ddof=1),
        "pooled_r2": pooled_metrics["r2"],
        "cv_mean_rmse": grp["rmse"].mean(),
        "pooled_rmse": pooled_metrics["rmse"],
        "cv_mean_spearman": grp["spearman"].mean(),
        "pooled_spearman": pooled_metrics["spearman"],
    })

reg_summary_df = pd.DataFrame(reg_summary_rows)
reg_summary_df.to_csv(REG_SUMMARY_PATH, index=False)
print("score_std_3way regression summary")
display(reg_summary_df.style.format({"cv_mean_r2": "{:.3f}", "cv_sd_r2": "{:.3f}", "pooled_r2": "{:.3f}", "cv_mean_rmse": "{:.3f}", "pooled_rmse": "{:.3f}", "cv_mean_spearman": "{:.3f}", "pooled_spearman": "{:.3f}"}))


In [ ]:
meeting_std3_tracking = (
    reg_pred_df.groupby(["bank", "doc_id", "date"], as_index=False)
    .agg(
        actual_std3_mean=("y_true_std3", "mean"),
        pred_std3_tfidf_mean=("pred_std3_tfidf", "mean"),
        pred_std3_gorodnichenko_mean=("pred_std3_gorodnichenko", "mean"),
        n_turns=("turn_uid", "size"),
    )
)
meeting_std3_tracking["meeting_date"] = pd.to_datetime(meeting_std3_tracking["date"].astype(str).str[:6], format="%Y%m", errors="coerce")
meeting_std3_tracking = meeting_std3_tracking.sort_values(["bank", "meeting_date", "doc_id"]).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, bank_label in zip(axes, ["Fed", "ECB", "BoE"]):
    sub = meeting_std3_tracking[meeting_std3_tracking["bank"] == bank_label].sort_values("meeting_date")
    ax.plot(sub["meeting_date"], sub["actual_std3_mean"], color="#222222", linewidth=1.6, marker="o", markersize=3, alpha=0.85, label="Actual mean std3")
    ax.plot(sub["meeting_date"], sub["pred_std3_tfidf_mean"], color="#01665e", linewidth=1.8, alpha=0.9, label="TF-IDF predicted")
    ax.plot(sub["meeting_date"], sub["pred_std3_gorodnichenko_mean"], color="#8c510a", linewidth=1.4, alpha=0.75, linestyle="--", label="Gorodnichenko predicted")
    ax.set_title(f"{bank_label}: mean score_std_3way")
    ax.set_xlabel("Meeting date")
    ax.grid(alpha=0.25)
    if bank_label == "Fed":
        ax.set_ylabel("Actual / predicted mean std3")
    ax.legend(frameon=True, fontsize=8)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_score_std3_timeseries.png", dpi=160, bbox_inches="tight")
finalize_plot()
display(meeting_std3_tracking.head())


In [ ]:
split_preds = pooled_pred_df[pooled_pred_df["target"] == "split"].copy()
bin_edges = np.linspace(0, 1, 11)
calibration_rows = []

for model_name, pred_col in [("Gorodnichenko", "pred_gorodnichenko"), ("TF-IDF", "pred_tfidf")]:
    tmp = split_preds[["y_true", pred_col]].copy().rename(columns={pred_col: "pred"})
    tmp["bin"] = pd.cut(tmp["pred"], bins=bin_edges, include_lowest=True, duplicates="drop")
    calib = (
        tmp.groupby("bin", observed=False, as_index=False)
        .agg(
            mean_pred=("pred", "mean"),
            actual_rate=("y_true", "mean"),
            n=("y_true", "size"),
        )
        .dropna(subset=["mean_pred", "actual_rate"])
    )
    calib["model"] = model_name
    calibration_rows.append(calib)

calibration_df = pd.concat(calibration_rows, ignore_index=True)

meeting_tracking = (
    split_preds.merge(
        disagreement[["turn_uid", "bank", "doc_id", "date"]],
        on=["turn_uid", "bank", "doc_id"],
        how="left",
    )
    .groupby(["bank", "doc_id", "date"], as_index=False)
    .agg(
        actual_split_share=("y_true", "mean"),
        pred_tfidf_share=("pred_tfidf", "mean"),
        pred_gorodnichenko_share=("pred_gorodnichenko", "mean"),
        n_turns=("turn_uid", "size"),
    )
)
meeting_tracking["meeting_date"] = pd.to_datetime(meeting_tracking["date"].astype(str).str[:6], format="%Y%m", errors="coerce")
meeting_tracking = meeting_tracking.sort_values(["bank", "meeting_date", "doc_id"]).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
calib_colors = {"Gorodnichenko": "#8c510a", "TF-IDF": "#01665e"}

for model_name in ["Gorodnichenko", "TF-IDF"]:
    sub = calibration_df[calibration_df["model"] == model_name]
    axes[0, 0].plot(sub["mean_pred"], sub["actual_rate"], marker="o", linewidth=2, color=calib_colors[model_name], label=model_name)

axes[0, 0].plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1.2, alpha=0.8, label="45-degree line")
axes[0, 0].set_title("Split calibration: predicted vs actual")
axes[0, 0].set_xlabel("Mean predicted split probability")
axes[0, 0].set_ylabel("Observed split rate")
axes[0, 0].set_xlim(0, 1)
axes[0, 0].set_ylim(0, 1)
axes[0, 0].legend(frameon=True, fontsize=8)
axes[0, 0].grid(alpha=0.25)

bank_order = ["Fed", "ECB", "BoE"]
bank_axes = {"Fed": axes[0, 1], "ECB": axes[1, 0], "BoE": axes[1, 1]}
for bank_label in bank_order:
    sub = meeting_tracking[meeting_tracking["bank"] == bank_label].sort_values("meeting_date")
    ax = bank_axes[bank_label]
    ax.plot(sub["meeting_date"], sub["actual_split_share"], color="#222222", linewidth=1.8, marker="o", markersize=3, alpha=0.9, label="Actual split share")
    ax.plot(sub["meeting_date"], sub["pred_tfidf_share"], color="#01665e", linewidth=1.6, alpha=0.85, label="TF-IDF predicted")
    ax.plot(sub["meeting_date"], sub["pred_gorodnichenko_share"], color="#8c510a", linewidth=1.4, alpha=0.75, linestyle="--", label="Gorodnichenko predicted")
    ax.set_title(f"{bank_label}: meeting-level split over time")
    ax.set_xlabel("Meeting date")
    ax.set_ylabel("Actual / predicted split share")
    ax.grid(alpha=0.25)
    ax.legend(frameon=True, fontsize=8)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_split_calibration_and_timeseries.png", dpi=160, bbox_inches="tight")
finalize_plot()
display(meeting_tracking.head())


## 8. TF-IDF Attribution: SHAP-Style Contributions and Top Terms

The environment does not include the `shap` package, so this notebook uses a stable linear-logit approximation instead: for the fitted TF-IDF logistic model, each feature contributes `tfidf_value × coefficient` to the log-odds. We use mean absolute contributions as a SHAP-style global importance summary, then keep the signed coefficient tables for exact term direction.


In [ ]:
feature_rows = []
top_frames = {}

split_model_full = clone(tfidf_clf).fit(disagreement["text"], disagreement["split"].values)
split_vec = split_model_full.named_steps["tfidf"]
split_logit = split_model_full.named_steps["logit"]
split_X = split_vec.transform(disagreement["text"])
split_coef = pd.Series(split_logit.coef_[0], index=split_vec.get_feature_names_out())
split_contrib = split_X.multiply(split_coef.values)
tfidf_shap_style = pd.DataFrame({
    "feature": split_coef.index,
    "coef": split_coef.values,
    "mean_abs_contrib": np.asarray(np.abs(split_contrib).mean(axis=0)).ravel(),
    "mean_signed_contrib": np.asarray(split_contrib.mean(axis=0)).ravel(),
}).sort_values("mean_abs_contrib", ascending=False)
tfidf_shap_style.to_csv(TFIDF_SHAP_STYLE_PATH, index=False)
top_shap_style = tfidf_shap_style.head(20).sort_values("mean_abs_contrib")
top_shap_style["direction"] = np.where(top_shap_style["coef"] >= 0, "Higher disagreement", "Lower disagreement")

for target in ["split", "high_std3"]:
    model = clone(tfidf_clf).fit(disagreement["text"], disagreement[target].values)
    vec = model.named_steps["tfidf"]
    logit = model.named_steps["logit"]
    coef = pd.Series(logit.coef_[0], index=vec.get_feature_names_out()).sort_values()

    coef_df = pd.DataFrame({
        "target": target,
        "feature": coef.index,
        "coef": coef.values,
    })
    coef_df["abs_coef"] = coef_df["coef"].abs()
    feature_rows.append(coef_df)

    top_negative = coef.head(15).sort_values(ascending=True).rename("coef").reset_index().rename(columns={"index": "feature"})
    top_negative["direction"] = "Lower disagreement"
    top_positive = coef.tail(15).sort_values(ascending=False).rename("coef").reset_index().rename(columns={"index": "feature"})
    top_positive["direction"] = "Higher disagreement"
    top_frames[target] = pd.concat([top_positive, top_negative], ignore_index=True)

tfidf_feature_df = pd.concat(feature_rows, ignore_index=True)
tfidf_feature_df.to_csv(TFIDF_FEATURES_PATH, index=False)

print("Top TF-IDF SHAP-style features for split")
display(top_shap_style[["feature", "direction", "coef", "mean_abs_contrib", "mean_signed_contrib"]].style.format({"coef": "{:.4f}", "mean_abs_contrib": "{:.5f}", "mean_signed_contrib": "{:.5f}"}))

for target in ["split", "high_std3"]:
    print(f"\nTop TF-IDF terms for {target}")
    display(top_frames[target].style.format({"coef": "{:.4f}"}))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False)
plot_targets = ["split", "high_std3"]
plot_titles = {"split": "Predicting split", "high_std3": "Predicting high_std3"}

for row_i, target in enumerate(plot_targets):
    frame = top_frames[target]
    high = frame[frame["direction"] == "Higher disagreement"].sort_values("coef")
    low = frame[frame["direction"] == "Lower disagreement"].sort_values("coef")

    axes[row_i, 0].barh(low["feature"], low["coef"], color="#377eb8")
    axes[row_i, 0].set_title(f"{plot_titles[target]}: lower disagreement", fontsize=11, fontweight="bold")
    axes[row_i, 0].set_xlabel("Logit coefficient")
    axes[row_i, 0].grid(axis="x", alpha=0.25)

    axes[row_i, 1].barh(high["feature"], high["coef"], color="#e41a1c")
    axes[row_i, 1].set_title(f"{plot_titles[target]}: higher disagreement", fontsize=11, fontweight="bold")
    axes[row_i, 1].set_xlabel("Logit coefficient")
    axes[row_i, 1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_tfidf_top_terms.png", dpi=160, bbox_inches="tight")
finalize_plot()

print(f"Saved TF-IDF feature table -> {TFIDF_FEATURES_PATH}")
print(f"Saved TF-IDF SHAP-style table -> {TFIDF_SHAP_STYLE_PATH}")
print(f"Saved TF-IDF term figure -> {FIGURE_DIR / '6_2_tfidf_top_terms.png'}")
print(f"Saved TF-IDF SHAP-style figure -> {FIGURE_DIR / '6_2_tfidf_shap_style.png'}")


## 9. Mechanism Check: Length, Mixed Signals, Inflation, Hedges, and Forward Guidance

This section asks whether disagreement is mostly a long-turn artifact or whether it is specifically associated with competing policy cues. The key hypothesis is that **mixed hawk and dove language in the same turn** raises disagreement, especially when inflation is part of that mixed cue bundle.


In [ ]:
try:
    import statsmodels.formula.api as smf
    HAVE_STATSMODELS = True
except Exception:
    HAVE_STATSMODELS = False

hawk_terms = {
    "inflation", "inflationary", "restrictive", "tighten", "tightening", "raise", "raised", "higher",
    "upside", "overheating", "wage", "wages", "pressures", "rates", "firming", "persistent"
}
dove_terms = {
    "support", "purchase", "purchases", "accommodative", "easing", "ease", "slowdown", "downside",
    "weaker", "unemployment", "cut", "lower", "stimulus", "softening", "cooling", "decline"
}
hedge_terms = {
    "may", "might", "could", "would", "perhaps", "possibly", "uncertain", "uncertainty", "risk", "risks",
    "monitor", "monitoring", "careful", "gradual", "depending", "conditional", "appropriate"
}
fg_terms = {
    "for some time", "data dependent", "data-dependent", "path", "pace", "gradual", "appropriate",
    "remain restrictive", "remain accommodative", "higher for longer", "until", "as long as"
}
growth_terms = {
    "growth", "activity", "demand", "employment", "job", "jobs", "labor", "labour",
    "slowdown", "weaker", "weakness", "decline", "cooling", "softening", "unemployment", "recession"
}
resolve_terms = {
    "determined", "resolve", "commitment", "committed", "ensure", "restore", "return",
    "target", "sufficiently", "necessary", "maintain", "until"
}
expectations_terms = {
    "expectation", "expectations", "anchored", "anchor", "credibility", "deanchoring", "deanchor",
    "longer", "term"
}
risk_uncertainty_terms = {
    "risk", "risks", "uncertain", "uncertainty", "monitor", "monitoring", "gradual",
    "conditional", "appropriate", "careful", "balanced", "balance"
}
noninfl_hawk_terms = {
    "restrictive", "tighten", "tightening", "raise", "raised", "higher", "upside",
    "overheating", "wage", "wages", "pressures", "rates", "firming", "persistent"
}

def count_prefix_matches(tokens: list[str], vocab: set[str]) -> int:
    return sum(any(tok.startswith(v) for v in vocab) for tok in tokens)

def count_phrase_matches(text: str, phrases: set[str]) -> int:
    low = str(text).lower()
    return sum(phrase in low for phrase in phrases)

def two_prop_test(success_a: int, n_a: int, success_b: int, n_b: int) -> dict[str, float]:
    p_a = success_a / n_a
    p_b = success_b / n_b
    pooled = (success_a + success_b) / (n_a + n_b)
    se_pool = np.sqrt(max(pooled * (1 - pooled) * (1 / n_a + 1 / n_b), 1e-12))
    z_stat = (p_b - p_a) / se_pool
    pvalue = 2 * (1 - norm.cdf(abs(z_stat)))
    diff = p_b - p_a
    se_diff = np.sqrt(max(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b, 1e-12))
    ci_low = diff - 1.96 * se_diff
    ci_high = diff + 1.96 * se_diff
    return {
        "rate_a": p_a,
        "rate_b": p_b,
        "diff": diff,
        "z_stat": z_stat,
        "pvalue": pvalue,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }

mech = disagreement.copy()
mech["tokens"] = mech["text"].astype(str).apply(tokenize)
mech["word_count"] = mech["tokens"].str.len().clip(lower=1)
mech["sentence_count"] = mech["text"].astype(str).str.count(r"[\.!?]") + 1
mech["hawk_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, hawk_terms))
mech["dove_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, dove_terms))
mech["hedge_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, hedge_terms))
mech["fg_count"] = mech["text"].apply(lambda txt: count_phrase_matches(txt, fg_terms))
mech["inflation_present"] = mech["tokens"].apply(lambda toks: int(any(tok.startswith("inflation") for tok in toks)))
mech["growth_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, growth_terms))
mech["resolve_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, resolve_terms))
mech["expectations_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, expectations_terms))
mech["risk_uncertainty_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, risk_uncertainty_terms))
mech["noninfl_hawk_count"] = mech["tokens"].apply(lambda toks: count_prefix_matches(toks, noninfl_hawk_terms))
mech["mixed_signal"] = ((mech["hawk_count"] > 0) & (mech["dove_count"] > 0)).astype(int)
mech["inflation_x_mixed"] = mech["inflation_present"] * mech["mixed_signal"]
mech["inflation_x_dovish"] = ((mech["inflation_present"] == 1) & (mech["dove_count"] > 0)).astype(int)
mech["inflation_x_other_hawk"] = ((mech["inflation_present"] == 1) & (mech["noninfl_hawk_count"] > 0)).astype(int)
mech["inflation_x_growth_concern"] = ((mech["inflation_present"] == 1) & (mech["growth_count"] > 0)).astype(int)
mech["inflation_x_resolve"] = ((mech["inflation_present"] == 1) & (mech["resolve_count"] > 0)).astype(int)
mech["inflation_x_expectations"] = ((mech["inflation_present"] == 1) & (mech["expectations_count"] > 0)).astype(int)
mech["inflation_x_risk_uncertainty"] = ((mech["inflation_present"] == 1) & (mech["risk_uncertainty_count"] > 0)).astype(int)
mech["hedge_rate"] = mech["hedge_count"] / mech["word_count"]
mech["fg_rate"] = mech["fg_count"] / mech["sentence_count"]
mech["word_count_z"] = (mech["word_count"] - mech["word_count"].mean()) / mech["word_count"].std(ddof=0)

display(mech[["split", "high_std3", "word_count", "hawk_count", "dove_count", "mixed_signal", "inflation_present", "hedge_rate", "fg_rate"]].head())
print(f"mixed_signal share: {mech['mixed_signal'].mean():.1%}")
print(f"inflation_present share: {mech['inflation_present'].mean():.1%}")


In [ ]:
# Simple mechanism exposure table: only direct features, no interaction terms

simple_exposure_specs = [
    ("mixed_signal", "Mixed hawk+dove language", "binary"),
    ("inflation_present", "Inflation/prices present", "binary"),
    ("hawk_count", "Any hawkish language", "count"),
    ("dove_count", "Any dovish language", "count"),
    ("hedge_count", "Any hedge/uncertainty language", "count"),
    ("fg_count", "Any forward-guidance language", "count"),
    ("growth_count", "Any growth/activity language", "count"),
    ("resolve_count", "Any resolve/commitment language", "count"),
    ("expectations_count", "Any expectations/credibility language", "count"),
    ("risk_uncertainty_count", "Any risk/uncertainty language", "count"),
]

simple_exposure_rows = []

for col, label, kind in simple_exposure_specs:
    if kind == "binary":
        present_mask = mech[col] == 1
    else:
        present_mask = mech[col] > 0

    absent = mech.loc[~present_mask, "split"]
    present = mech.loc[present_mask, "split"]

    simple_exposure_rows.append({
        "exposure": label,
        "absent_n": int(len(absent)),
        "present_n": int(len(present)),
        "absent_split_rate": absent.mean(),
        "present_split_rate": present.mean(),
        "diff_pp": 100 * (present.mean() - absent.mean()),
    })

simple_exposure_table = pd.DataFrame(simple_exposure_rows)

display(
    simple_exposure_table.style.format({
        "absent_split_rate": "{:.1%}",
        "present_split_rate": "{:.1%}",
        "diff_pp": "{:+.1f}",
    })
)

In [ ]:
from scipy.stats import norm

def two_prop_ci(success_a, n_a, success_b, n_b):
    p_a = success_a / n_a
    p_b = success_b / n_b
    diff = p_b - p_a
    se = np.sqrt(max(
        p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b,
        1e-12
    ))
    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se

    pooled = (success_a + success_b) / (n_a + n_b)
    se_pool = np.sqrt(max(
        pooled * (1 - pooled) * (1 / n_a + 1 / n_b),
        1e-12
    ))
    z_stat = diff / se_pool
    pvalue = 2 * (1 - norm.cdf(abs(z_stat)))
    return diff, ci_low, ci_high, pvalue

simple_exposure_specs = [
    ("mixed_signal", "Mixed hawk+dove language", "binary"),
    ("inflation_present", "Inflation/prices present", "binary"),
    ("hawk_count", "Any hawkish language", "count"),
    ("dove_count", "Any dovish language", "count"),
    ("hedge_count", "Any hedge/uncertainty language", "count"),
    ("fg_count", "Any forward-guidance language", "count"),
    ("growth_count", "Any growth/activity language", "count"),
    ("resolve_count", "Any resolve/commitment language", "count"),
    ("expectations_count", "Any expectations/credibility language", "count"),
    ("risk_uncertainty_count", "Any risk/uncertainty language", "count"),
]

simple_exposure_rows = []

for col, label, kind in simple_exposure_specs:
    present_mask = mech[col].eq(1) if kind == "binary" else mech[col].gt(0)

    absent = mech.loc[~present_mask, "split"]
    present = mech.loc[present_mask, "split"]

    success_a = int(absent.sum())
    n_a = int(len(absent))
    success_b = int(present.sum())
    n_b = int(len(present))

    diff, ci_low, ci_high, pvalue = two_prop_ci(success_a, n_a, success_b, n_b)

    simple_exposure_rows.append({
        "exposure": label,
        "absent_n": n_a,
        "present_n": n_b,
        "absent_split_rate": success_a / n_a,
        "present_split_rate": success_b / n_b,
        "diff_pp": 100 * diff,
        "ci_low_pp": 100 * ci_low,
        "ci_high_pp": 100 * ci_high,
        "pvalue": pvalue,
        "diff_ci": f"{100*diff:+.1f} pp [{100*ci_low:.1f}, {100*ci_high:.1f}]",
    })

simple_exposure_table = pd.DataFrame(simple_exposure_rows)

display(
    simple_exposure_table[
        [
            "exposure",
            "absent_n",
            "present_n",
            "absent_split_rate",
            "present_split_rate",
            "diff_ci",
            "pvalue",
        ]
    ].style.format({
        "absent_split_rate": "{:.1%}",
        "present_split_rate": "{:.1%}",
        "pvalue": "{:.3g}",
    })
)

In [ ]:
length_diag = (
    mech.assign(word_decile=pd.qcut(mech["word_count"], q=10, labels=False, duplicates="drop"))
    .groupby("word_decile", as_index=False)
    .agg(
        mean_words=("word_count", "mean"),
        split_rate=("split", "mean"),
        high_std3_rate=("high_std3", "mean"),
    )
)

mixed_diag = (
    mech.groupby("mixed_signal", as_index=False)
    .agg(
        n_turns=("turn_uid", "size"),
        mean_words=("word_count", "mean"),
        split_rate=("split", "mean"),
        high_std3_rate=("high_std3", "mean"),
        inflation_share=("inflation_present", "mean"),
    )
)
mixed_diag["mixed_signal"] = mixed_diag["mixed_signal"].map({0: "No mixed signal", 1: "Mixed hawk+dove"})

exposure_specs = [
    ("mixed_signal", {0: "No mixed signal", 1: "Mixed hawk+dove"}),
    ("inflation_present", {0: "No inflation/prices", 1: "Inflation/prices present"}),
]
exposure_tables = []
exposure_test_rows = []
for feature, labels in exposure_specs:
    counts = mech.groupby(feature)["split"].agg(["sum", "count"]).reset_index()
    counts = counts.sort_values(feature).reset_index(drop=True)
    test_res = two_prop_test(
        int(counts.loc[0, "sum"]),
        int(counts.loc[0, "count"]),
        int(counts.loc[1, "sum"]),
        int(counts.loc[1, "count"]),
    )
    exposure_test_rows.append({
        "feature": feature,
        "baseline_group": labels[counts.loc[0, feature]],
        "comparison_group": labels[counts.loc[1, feature]],
        "split_rate_diff": test_res["diff"],
        "ci_low": test_res["ci_low"],
        "ci_high": test_res["ci_high"],
        "z_stat": test_res["z_stat"],
        "pvalue": test_res["pvalue"],
    })
    tmp = (
        mech.groupby(feature, as_index=False)
        .agg(
            n_turns=("turn_uid", "size"),
            split_rate=("split", "mean"),
            high_std3_rate=("high_std3", "mean"),
            mean_std3=("score_std_3way", "mean"),
            mean_words=("word_count", "mean"),
        )
        .rename(columns={feature: "group_value"})
    )
    tmp["feature"] = feature
    tmp["group"] = tmp["group_value"].map(labels)
    tmp["non_split_rate"] = 1 - tmp["split_rate"]
    tmp["not_high_std3_rate"] = 1 - tmp["high_std3_rate"]
    exposure_tables.append(tmp[["feature", "group", "n_turns", "split_rate", "non_split_rate", "high_std3_rate", "not_high_std3_rate", "mean_std3", "mean_words"]])

mechanism_exposure_df = pd.concat(exposure_tables, ignore_index=True)
mechanism_exposure_tests = pd.DataFrame(exposure_test_rows)
mechanism_exposure_df.to_csv(MECH_EXPOSURE_PATH, index=False)
mechanism_exposure_tests.to_csv(MECH_EXPOSURE_TEST_PATH, index=False)

print("Length deciles")
display(length_diag.style.format({"mean_words": "{:.1f}", "split_rate": "{:.1%}", "high_std3_rate": "{:.1%}"}))
print("\nMechanism exposure table")
display(mechanism_exposure_df.style.format({"split_rate": "{:.1%}", "non_split_rate": "{:.1%}", "high_std3_rate": "{:.1%}", "not_high_std3_rate": "{:.1%}", "mean_std3": "{:.3f}", "mean_words": "{:.1f}"}))
print("\nTwo-proportion tests for split rates")
display(mechanism_exposure_tests.style.format({"split_rate_diff": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "z_stat": "{:.2f}", "pvalue": "{:.3g}"}))
print("\nMixed-signal descriptive table")
display(mixed_diag.style.format({"mean_words": "{:.1f}", "split_rate": "{:.1%}", "high_std3_rate": "{:.1%}", "inflation_share": "{:.1%}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(length_diag["mean_words"], length_diag["split_rate"], marker="o", color="#1b9e77", label="split")
axes[0].plot(length_diag["mean_words"], length_diag["high_std3_rate"], marker="o", color="#d95f02", label="high_std3")
axes[0].set_title("Disagreement by turn length decile")
axes[0].set_xlabel("Mean words in decile")
axes[0].set_ylabel("Rate")
axes[0].legend(frameon=True)
axes[0].grid(alpha=0.25)

x = np.arange(len(mixed_diag))
w = 0.35
axes[1].bar(x - w/2, mixed_diag["split_rate"], width=w, color="#1b9e77", label="split")
axes[1].bar(x + w/2, mixed_diag["high_std3_rate"], width=w, color="#d95f02", label="high_std3")
axes[1].set_xticks(x)
axes[1].set_xticklabels(mixed_diag["mixed_signal"])
axes[1].set_title("Mixed-signal turns are more disagreement-prone")
axes[1].set_ylabel("Rate")
axes[1].legend(frameon=True)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_mechanism_length_mixed.png", dpi=160, bbox_inches="tight")
finalize_plot()


In [ ]:
keep_terms = ["mixed_signal", "inflation_present", "inflation_x_mixed", "hedge_rate", "fg_rate", "word_count_z"]
mechanism_df = pd.DataFrame()

if HAVE_STATSMODELS:
    formula = "TARGET ~ mixed_signal + inflation_present + inflation_x_mixed + hedge_rate + fg_rate + word_count_z + C(bank)"
    mech_rows = []
    for target in ["split", "high_std3"]:
        fit = smf.logit(formula.replace("TARGET", target), data=mech).fit(disp=False)
        params = fit.params
        pvals = fit.pvalues
        conf = fit.conf_int()
        for term in keep_terms:
            mech_rows.append({
                "target": target,
                "term": term,
                "coef": params[term],
                "odds_ratio": float(np.exp(params[term])),
                "pvalue": pvals[term],
                "ci_low": conf.loc[term, 0],
                "ci_high": conf.loc[term, 1],
                "method": "statsmodels_logit",
            })
    mechanism_df = pd.DataFrame(mech_rows)
else:
    feature_cols = keep_terms + ["bank"]
    X = pd.get_dummies(mech[feature_cols], columns=["bank"], drop_first=True, dtype=float)
    mech_rows = []
    for target in ["split", "high_std3"]:
        model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)
        model.fit(X, mech[target].values)
        coef_map = pd.Series(model.coef_[0], index=X.columns)
        for term in keep_terms:
            mech_rows.append({
                "target": target,
                "term": term,
                "coef": float(coef_map[term]),
                "odds_ratio": float(np.exp(coef_map[term])),
                "pvalue": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
                "method": "sklearn_logit_no_se",
            })
    mechanism_df = pd.DataFrame(mech_rows)
    print("statsmodels not available; using sklearn logistic fallback without p-values")

mechanism_df.to_csv(MECH_TABLE_PATH, index=False)
for target in ["split", "high_std3"]:
    print(f"\nMechanism logit: {target}")
    cols = ["term", "coef", "odds_ratio", "pvalue", "ci_low", "ci_high", "method"]
    display(mechanism_df.loc[mechanism_df["target"] == target, cols].style.format({"coef": "{:.3f}", "odds_ratio": "{:.3f}", "pvalue": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}"}))


## 10. Inflation Framing Deep Dive

Because inflation language is a dominant predictor of disagreement, this section asks whether disagreement rises when inflation is embedded in different macro frames. The goal is not to say that inflation alone causes disagreement, but to test whether specific inflation co-occurrences --- dovish language, other hawkish language, growth concerns, resolve/tightening, expectations, and risk/uncertainty --- change how likely models are to move off neutral.


In [ ]:
inflation_frame_specs = [
    ("inflation_x_dovish", "Inflation x dovish terms"),
    ("inflation_x_other_hawk", "Inflation x other hawkish terms"),
    ("inflation_x_growth_concern", "Inflation x growth concerns"),
    ("inflation_x_resolve", "Inflation x resolve/tightening"),
    ("inflation_x_expectations", "Inflation x expectations"),
    ("inflation_x_risk_uncertainty", "Inflation x risk/uncertainty"),
]

infl_sub = mech[mech["inflation_present"] == 1].copy()
infl_rows = []
infl_test_rows = []
for feature, label in inflation_frame_specs:
    grouped = (
        infl_sub.groupby(feature, as_index=False)
        .agg(
            n_turns=("turn_uid", "size"),
            split_rate=("split", "mean"),
            high_std3_rate=("high_std3", "mean"),
            mean_std3=("score_std_3way", "mean"),
            mean_words=("word_count", "mean"),
        )
    )
    grouped["frame"] = label
    grouped["group"] = grouped[feature].map({0: "Absent within inflation turns", 1: "Present within inflation turns"})
    infl_rows.append(grouped[["frame", "group", "n_turns", "split_rate", "high_std3_rate", "mean_std3", "mean_words"]])

    counts = infl_sub.groupby(feature)["split"].agg(["sum", "count"]).reset_index().sort_values(feature).reset_index(drop=True)
    test_res = two_prop_test(
        int(counts.loc[0, "sum"]),
        int(counts.loc[0, "count"]),
        int(counts.loc[1, "sum"]),
        int(counts.loc[1, "count"]),
    )
    infl_test_rows.append({
        "frame": label,
        "split_rate_diff": test_res["diff"],
        "ci_low": test_res["ci_low"],
        "ci_high": test_res["ci_high"],
        "pvalue": test_res["pvalue"],
    })

inflation_frame_df = pd.concat(infl_rows, ignore_index=True)
inflation_frame_tests = pd.DataFrame(infl_test_rows)
inflation_frame_df.to_csv(INFLATION_FRAME_PATH, index=False)

model_frames = []
for llm, fp in INPUT_FILES.items():
    tmp = pd.read_csv(fp, usecols=["turn_uid", "label"])
    tmp["label"] = tmp["label"].astype(str).str.strip().str.lower()
    tmp["model"] = llm
    tmp["is_stanced"] = tmp["label"].isin(STANCED).astype(int)
    model_frames.append(tmp)

model_labels = pd.concat(model_frames, ignore_index=True)
infl_model = infl_sub[["turn_uid"] + [f for f, _ in inflation_frame_specs]].merge(model_labels, on="turn_uid", how="left")

model_rows = []
for feature, label in inflation_frame_specs:
    sub = infl_model[infl_model[feature] == 1].copy()
    grp = (
        sub.groupby("model", as_index=False)
        .agg(
            n_turns=("turn_uid", "size"),
            stanced_rate=("is_stanced", "mean"),
        )
    )
    grp["neutral_rate"] = 1 - grp["stanced_rate"]
    grp["frame"] = label
    model_rows.append(grp[["frame", "model", "n_turns", "stanced_rate", "neutral_rate"]])

inflation_model_df = pd.concat(model_rows, ignore_index=True)
inflation_model_df.to_csv(INFLATION_MODEL_PATH, index=False)

print("Inflation framing: overall disagreement within inflation turns")
display(inflation_frame_df.style.format({"split_rate": "{:.1%}", "high_std3_rate": "{:.1%}", "mean_std3": "{:.3f}", "mean_words": "{:.1f}"}))
print("\nInflation framing: split-rate differences within inflation turns")
display(inflation_frame_tests.style.format({"split_rate_diff": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}", "pvalue": "{:.3g}"}))
print("\nInflation framing by model: stanced rate within present frame")
display(inflation_model_df.style.format({"stanced_rate": "{:.1%}", "neutral_rate": "{:.1%}"}))

present_rows = inflation_frame_df[inflation_frame_df["group"] == "Present within inflation turns"].copy()
heat = inflation_model_df.pivot(index="frame", columns="model", values="stanced_rate")
heat = heat.reindex(index=[label for _, label in inflation_frame_specs])
heat = heat[[c for c in LLMS if c in heat.columns]]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(present_rows["frame"], present_rows["split_rate"], color="#2a9d8f")
axes[0].set_title("Split rate within inflation turns")
axes[0].set_xlabel("Split rate")
axes[0].grid(axis="x", alpha=0.25)

im = axes[1].imshow(heat.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
axes[1].set_title("Model stanced rate by inflation frame")
axes[1].set_xticks(np.arange(len(heat.columns)))
axes[1].set_xticklabels(heat.columns, rotation=45, ha="right")
axes[1].set_yticks(np.arange(len(heat.index)))
axes[1].set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        axes[1].text(j, i, f"{val:.2f}", ha="center", va="center", color="black", fontsize=8)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_frames.png", dpi=160, bbox_inches="tight")
finalize_plot()


In [ ]:
frame_count_specs = [
    ("dove_count", "Inflation x dovish terms"),
    ("noninfl_hawk_count", "Inflation x other hawkish terms"),
    ("growth_count", "Inflation x growth concerns"),
    ("resolve_count", "Inflation x resolve/tightening"),
    ("expectations_count", "Inflation x expectations"),
    ("risk_uncertainty_count", "Inflation x risk/uncertainty"),
]

def make_density_bins(series, n_quantiles=8):
    series = pd.Series(series)
    out = pd.Series(np.nan, index=series.index, dtype=object)
    zero_mask = series <= 0
    out.loc[zero_mask] = "zero"
    pos = series.loc[~zero_mask]
    if pos.empty:
        return out
    q = min(n_quantiles, max(1, pos.nunique()))
    if q == 1:
        out.loc[~zero_mask] = "q1"
        return out
    ranks = pos.rank(method="first")
    labels = [f"q{i}" for i in range(1, q + 1)]
    out.loc[~zero_mask] = pd.qcut(ranks, q=q, labels=labels)
    return out

dose_rows = []
dose_model_rows = []
score3_map = {"dovish": -1, "mostly dovish": -1, "neutral": 0, "mostly hawkish": 1, "hawkish": 1}

infl_model_full = infl_sub[["turn_uid", "word_count"] + [c for c, _ in frame_count_specs]].merge(model_labels[["turn_uid", "model", "label", "is_stanced"]], on="turn_uid", how="left")
infl_model_full["score3"] = infl_model_full["label"].map(score3_map)
infl_model_full["abs_score3"] = infl_model_full["score3"].abs()

for count_col, frame_label in frame_count_specs:
    tmp = infl_sub[["turn_uid", "split", "score_std_3way", "word_count", count_col]].copy()
    tmp["term_rate_per_100"] = 100 * tmp[count_col] / tmp["word_count"].clip(lower=1)
    tmp["intensity_bin"] = make_density_bins(tmp["term_rate_per_100"], n_quantiles=8)
    pooled = (
        tmp.groupby("intensity_bin", as_index=False)
        .agg(
            n_turns=("turn_uid", "size"),
            split_rate=("split", "mean"),
            mean_std3=("score_std_3way", "mean"),
            mean_term_rate=("term_rate_per_100", "mean"),
        )
    )
    pooled["frame"] = frame_label
    dose_rows.append(pooled[["frame", "intensity_bin", "n_turns", "split_rate", "mean_std3", "mean_term_rate"]])

    tmp_model = infl_model_full[["turn_uid", "model", "is_stanced", "abs_score3", "word_count", count_col]].copy()
    tmp_model["term_rate_per_100"] = 100 * tmp_model[count_col] / tmp_model["word_count"].clip(lower=1)
    tmp_model["intensity_bin"] = make_density_bins(tmp_model["term_rate_per_100"], n_quantiles=8)
    by_model = (
        tmp_model.groupby(["model", "intensity_bin"], as_index=False)
        .agg(
            n_turns=("turn_uid", "size"),
            stanced_rate=("is_stanced", "mean"),
            mean_abs_score3=("abs_score3", "mean"),
            mean_term_rate=("term_rate_per_100", "mean"),
        )
    )
    by_model["frame"] = frame_label
    dose_model_rows.append(by_model[["frame", "model", "intensity_bin", "n_turns", "stanced_rate", "mean_abs_score3", "mean_term_rate"]])

inflation_dose_df = pd.concat(dose_rows, ignore_index=True)
inflation_model_dose_df = pd.concat(dose_model_rows, ignore_index=True)
model_baseline = model_labels.groupby("model", as_index=False)["is_stanced"].mean().rename(columns={"is_stanced": "baseline_stanced_rate"})
inflation_model_dose_df = inflation_model_dose_df.merge(model_baseline, on="model", how="left")
inflation_model_dose_df["stanced_rate_centered"] = inflation_model_dose_df["stanced_rate"] - inflation_model_dose_df["baseline_stanced_rate"]
inflation_model_dose_df["nonneutral_rate"] = inflation_model_dose_df["stanced_rate"]
inflation_dose_df.to_csv(INFLATION_DOSERESP_PATH, index=False)
inflation_model_dose_df.to_csv(INFLATION_MODEL_DOSERESP_PATH, index=False)
inflation_model_dose_df.to_csv(INFLATION_MODEL_CENTERED_PATH, index=False)

model_colors = {
    "deepseekv3": "#1f77b4",
    "gemini25flash": "#ff7f0e",
    "gpt-4o": "#2ca02c",
    "llama33": "#d62728",
    "mistrallarge_or": "#9467bd",
    "qwen25_72b": "#8c564b",
}

fig, axes = plt.subplots(3, 2, figsize=(15, 13), sharex=False, sharey=True)
for ax, frame_label in zip(axes.flat, [label for _, label in frame_count_specs]):
    sub = inflation_dose_df[inflation_dose_df["frame"] == frame_label].copy()
    sub = sub.sort_values("mean_term_rate")
    ax.plot(sub["mean_term_rate"], sub["split_rate"], marker="o", linewidth=2.0, color="#2a9d8f", label="Split rate")
    ax.plot(sub["mean_term_rate"], sub["mean_std3"], marker="s", linewidth=1.9, color="#264653", label="Mean std3")
    ax.set_title(frame_label)
    ax.set_xlabel("Mean co-occurring term rate per 100 words")
    ax.set_ylabel("Outcome")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend(frameon=True, fontsize=8)

plt.suptitle("Inflation co-occurrence dose response: pooled disagreement", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_dose_response_pooled.png", dpi=160, bbox_inches="tight")
finalize_plot()

fig, axes = plt.subplots(3, 2, figsize=(15, 13), sharex=False, sharey=True)
for ax, frame_label in zip(axes.flat, [label for _, label in frame_count_specs]):
    sub = inflation_model_dose_df[inflation_model_dose_df["frame"] == frame_label].copy()
    for model_name in LLMS:
        tmp = sub[sub["model"] == model_name].copy()
        if tmp.empty:
            continue
        tmp = tmp.sort_values("mean_term_rate")
        ax.plot(tmp["mean_term_rate"], tmp["stanced_rate"], marker="o", linewidth=1.8, color=model_colors[model_name], label=model_name)
    ax.set_title(frame_label)
    ax.set_xlabel("Mean co-occurring term rate per 100 words")
    ax.set_ylabel("Model stanced rate")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)

handles = [plt.Line2D([0], [0], color=model_colors[m], marker='o', linewidth=1.8, label=m) for m in LLMS]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("Inflation co-occurrence dose response: model stanced rates", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_dose_response_models.png", dpi=160, bbox_inches="tight")
finalize_plot()

fig, axes = plt.subplots(3, 2, figsize=(15, 13), sharex=False, sharey=True)
for ax, frame_label in zip(axes.flat, [label for _, label in frame_count_specs]):
    sub = inflation_model_dose_df[inflation_model_dose_df["frame"] == frame_label].copy()
    for model_name in LLMS:
        tmp = sub[sub["model"] == model_name].copy()
        if tmp.empty:
            continue
        tmp = tmp.sort_values("mean_term_rate")
        ax.plot(tmp["mean_term_rate"], tmp["stanced_rate_centered"], marker="o", linewidth=1.8, color=model_colors[model_name], label=model_name)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.8)
    ax.set_title(frame_label)
    ax.set_xlabel("Mean co-occurring term rate per 100 words")
    ax.set_ylabel("Centered stanced rate")
    ax.grid(alpha=0.25)

fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("Inflation co-occurrence dose response: centered model stance", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_dose_response_centered.png", dpi=160, bbox_inches="tight")
finalize_plot()

fig, axes = plt.subplots(3, 2, figsize=(15, 13), sharex=False, sharey=True)
for ax, frame_label in zip(axes.flat, [label for _, label in frame_count_specs]):
    sub = inflation_model_dose_df[inflation_model_dose_df["frame"] == frame_label].copy()
    for model_name in LLMS:
        tmp = sub[sub["model"] == model_name].copy()
        if tmp.empty:
            continue
        tmp = tmp.sort_values("mean_term_rate")
        ax.plot(tmp["mean_term_rate"], tmp["nonneutral_rate"], marker="o", linewidth=1.8, color=model_colors[model_name], label=model_name)
    ax.set_title(frame_label)
    ax.set_xlabel("Mean co-occurring term rate per 100 words")
    ax.set_ylabel("Non-neutral rate")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)

fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("Inflation co-occurrence dose response: raw non-neutral rates", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_dose_response_nonneutral.png", dpi=160, bbox_inches="tight")
finalize_plot()

fig, axes = plt.subplots(3, 2, figsize=(15, 13), sharex=False, sharey=True)
for ax, frame_label in zip(axes.flat, [label for _, label in frame_count_specs]):
    sub = inflation_model_dose_df[inflation_model_dose_df["frame"] == frame_label].copy()
    for model_name in LLMS:
        tmp = sub[sub["model"] == model_name].copy()
        if tmp.empty:
            continue
        tmp = tmp.sort_values("mean_term_rate")
        ax.plot(tmp["mean_term_rate"], tmp["mean_abs_score3"], marker="o", linewidth=1.8, color=model_colors[model_name], label=model_name)
    ax.set_title(frame_label)
    ax.set_xlabel("Mean co-occurring term rate per 100 words")
    ax.set_ylabel("Mean absolute score3")
    ax.grid(alpha=0.25)

fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.01))
plt.suptitle("Inflation co-occurrence dose response: model score magnitude", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "6_2_inflation_dose_response_model_score.png", dpi=160, bbox_inches="tight")
finalize_plot()
print("Inflation dose-response tables")
display(inflation_dose_df.style.format({"split_rate": "{:.1%}", "mean_std3": "{:.3f}", "mean_term_rate": "{:.2f}"}))


## 11. Final Framing

Recommended reading of the results:

- **Headline target**: `split`
  - best for the substantive claim that disagreement lives at the neutral-vs-stanced boundary
- **Secondary target**: `high_std3`
  - best for unusually large instability once the same `{-1, 0, +1}` label scale is imposed on all models

If TF-IDF materially outperforms the Gorodnichenko dictionary, the interpretation is not that the dictionary is wrong. It means a fixed phrase list captures some policy content, but disagreement depends on contextual composition that bag-of-ngrams can exploit more flexibly.


In [ ]:
split_tbl = summary_df[summary_df["target"] == "split"].set_index("model")
std_tbl = summary_df[summary_df["target"] == "high_std3"].set_index("model")

print("Stable takeaway template:\n")
print(
    f"Using meeting-grouped 5-fold CV, TF-IDF reaches a pooled ROC-AUC of {split_tbl.loc['TF-IDF', 'pooled_headline']:.3f} "
    f"for the headline split target, compared with {split_tbl.loc['Gorodnichenko', 'pooled_headline']:.3f} for the Gorodnichenko dictionary."
)
print(
    f"On the high-instability target high_std3, TF-IDF reaches a pooled ROC-AUC of {std_tbl.loc['TF-IDF', 'pooled_headline']:.3f}, "
    f"compared with {std_tbl.loc['Gorodnichenko', 'pooled_headline']:.3f} for the dictionary baseline."
)
print(
    f"For the same high_std3 target, TF-IDF reaches a pooled PR-AUC of {std_tbl.loc['TF-IDF', 'pooled_secondary']:.3f}, "
    f"compared with {std_tbl.loc['Gorodnichenko', 'pooled_secondary']:.3f} for the dictionary baseline."
)
reg_tbl = reg_summary_df.set_index("model")
print(
    f"When disagreement intensity is modeled continuously through score_std_3way, TF-IDF reaches a pooled Spearman correlation of {reg_tbl.loc['TF-IDF', 'pooled_spearman']:.3f} "
    f"versus {reg_tbl.loc['Gorodnichenko', 'pooled_spearman']:.3f} for the dictionary baseline."
)
print("\nFiles written:")
for path in [
    SUMMARY_PATH,
    PRAUC_TABLE_PATH,
    FOLD_METRICS_PATH,
    POOLED_PRED_PATH,
    REG_SUMMARY_PATH,
    REG_PRED_PATH,
    TFIDF_FEATURES_PATH,
    TFIDF_SHAP_STYLE_PATH,
    MECH_TABLE_PATH,
    MECH_EXPOSURE_PATH,
    MECH_EXPOSURE_TEST_PATH,
    INFLATION_FRAME_PATH,
    INFLATION_MODEL_PATH,
    INFLATION_DOSERESP_PATH,
    INFLATION_MODEL_DOSERESP_PATH,
    INFLATION_MODEL_CENTERED_PATH,
    FIGURE_DIR / '6_2_gorodnichenko_vs_tfidf.png',
    FIGURE_DIR / '6_2_split_calibration_and_timeseries.png',
    FIGURE_DIR / '6_2_score_std3_timeseries.png',
    FIGURE_DIR / '6_2_inflation_frames.png',
    FIGURE_DIR / '6_2_inflation_dose_response_pooled.png',
    FIGURE_DIR / '6_2_inflation_dose_response_models.png',
    FIGURE_DIR / '6_2_inflation_dose_response_model_score.png',
    FIGURE_DIR / '6_2_inflation_dose_response_centered.png',
    FIGURE_DIR / '6_2_inflation_dose_response_nonneutral.png',
    FIGURE_DIR / '6_2_tfidf_top_terms.png',
    FIGURE_DIR / '6_2_tfidf_shap_style.png',
    FIGURE_DIR / '6_2_mechanism_length_mixed.png',
]:
    print(f"  -> {path.relative_to(ROOT)}")
